# Compiled cost on the card: K=64 candidates and so3c_knn_set at K=128

Per-epoch estimates (batch 256) with `--compile`, synthetic jets, T4 with
torch 2.11.0+cu128 pinned: K=64: pairlab, pairlat8lab, w64s24lab, pairlat8w64s24lab, knn16; K=128: pairlab, knn8, knn16, knn32.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
PROFILE = [(64, 1024, ['pairlab', 'pairlat8lab', 'w64s24lab', 'pairlat8w64s24lab', 'knn16']), (128, 512, ['pairlab', 'knn8', 'knn16', 'knn32'])]
for k, chunk, configs in PROFILE:
    args = ["benchmarks.profile_gpu", "--out", "/kaggle/working/knn_profile_k%d" % k,
            "--k", str(k), "--batch-size", "256", "--eval-chunk-size", str(chunk),
            "--compile", "--no-profile", "--steps", "15"]
    for c in configs:
        args += ["--config", c]
    ok = run(args)
    assert ok, "profile at K=%d failed" % k

In [ ]:
import json
for k, _, _ in PROFILE:
    s = json.load(open("/kaggle/working/knn_profile_k%d/summary.json" % k))
    print("K=%d on %s, torch %s" % (k, s["device"], s["torch"]))
    for r in s["rows"]:
        if "compile_error" in r:
            print("  %-18s compile error" % r["config"]); continue
        print("  %-18s %6d params | epoch ~%5.0f s (%.1f h for 60) | peak %.2f GiB%s"
              % (r["config"], r["n_params"], r["epoch_estimate_s"], r["epoch_estimate_s"] * 60 / 3600,
                 r["peak_train_gib"], r["compile_note"]))